In [6]:
import numpy as np
import pandas as pd
import warnings
import light_curve as lc
import optuna

from catboost import CatBoostClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from tqdm import tqdm, TqdmWarning
from datetime import datetime
from pathlib import Path
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from typing import Literal, Callable, Any


warnings.filterwarnings("ignore", category=TqdmWarning)

In [7]:
Type = Literal["train", "test"]
Split = Literal["split_01", "split_02", "split_03", "split_04", "split_05", "split_06", "split_07", "split_08", "split_09", "split_10", "split_11", "split_12", "split_13", "split_14", "split_15", "split_16", "split_17", "split_18", "split_19", "split_20"]


EPS = np.finfo(float).eps


def now() -> str:
    return datetime.now().astimezone().strftime("%Y%m%d-%H%M%S-%z")

### Data Loading

In [8]:
def load_log_df(type: Type, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{type}_log.parquet", **kwargs)

def load_flc_df(type: Type, split: Split, **kwargs) -> pd.DataFrame:
    return pd.read_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet", **kwargs)


def __ingest_dfs(type: Type):
    log_df = pd.read_csv(f"../artifacts/kaggle/{type}_log.csv", index_col="object_id")
    log_df.to_parquet(f"../artifacts/kaggle/{type}_log.parquet")

    splits = sorted(log_df["split"].unique())
    for split in splits:
        flc_df = pd.read_csv(f"../artifacts/kaggle/{split}/{type}_full_lightcurves.csv")
        flc_df.to_parquet(f"../artifacts/kaggle/{split}/{type}_flc.parquet")


__ingest_dfs(type="train")
__ingest_dfs(type="test")

### Feature Engineering

In [9]:
def load_all_feats_df(type: Type, **kwargs):
    return pd.concat([pd.read_parquet(filepath, **kwargs) for filepath in Path("../artifacts/feats").rglob(f"{type}_feats.parquet")])

def load_feats_df(type: Type, split: Split, **kwargs):
    return pd.read_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet", **kwargs)


def __build_and_ingest_feats(type: Type):
    log_df = load_log_df(type=type)

    with tqdm(total=len(log_df), unit="obj") as pb:
        for split, log_sub_df in log_df.groupby("split"):
            pb.set_description(f"Building features for `{split}` (`{type}`)")

            feats_buf = []

            for obj_id, log_row in log_sub_df.iterrows():
                flc_df = load_flc_df(type=type, split=split, filters=[("object_id", "==", obj_id)]) # pyright: ignore[reportArgumentType]
                feats = __build_feats_for_obj(log_row, flc_df)
                feats_buf.append(feats)

                pb.update()

            Path(f"../artifacts/feats/{split}").mkdir(parents=True, exist_ok=True)

            pd.DataFrame(feats_buf, index=log_sub_df.index).to_parquet(f"../artifacts/feats/{split}/{type}_feats.parquet")


# TODO Add features acquired from domain knowledge
def __build_feats_for_obj(log_row: pd.Series, flc_df: pd.DataFrame) -> dict:
    # Could reorder?
    __de_extinct(log_row, flc_df)

    flc_df["Flux_ratio"] = flc_df["Flux"] / flc_df["Flux_err"]

    feats = log_row.to_dict()

    feats.update(__build_stats_feats_for_obj(flc_df))
    feats.update(__build_lc_feats_for_obj(flc_df))
    feats.update(__build_domain_feats_for_obj(flc_df))

    return feats


def __build_stats_feats_for_obj(flc_df: pd.DataFrame) -> dict:
    feats = {}

    pivot_flc_df = flc_df.pivot_table(index="Time (MJD)", columns="Filter", values=["Flux", "Flux_err", "Flux_ratio"])

    for agg_name, agg in __STATS_AGGS.items():
        for feat in ["Flux", "Flux_err", "Flux_ratio"]:
            feats[f"{feat}_{agg_name}"] = agg(flc_df[feat])
            
            for filter in __FILTERS:
                if filter in pivot_flc_df.columns:
                    feats[f"{feat}_{agg_name}_{filter}"] = agg(pivot_flc_df[(feat, filter)]) # pyright: ignore[reportCallIssue]

    return feats


__STATS_AGGS: dict[str, Callable[[pd.Series], Any]] = {
    "mean": np.mean,
    "std": np.std,
    "min": np.min,
    "max": np.max,
    "median": np.median,
    "q25": lambda feats: feats.quantile(0.25),
    "q75": lambda feats: feats.quantile(0.75),
}
__FILTERS = ["u", "g", "r", "i", "z", "y"]


def __build_lc_feats_for_obj(flc_df: pd.DataFrame) -> dict:   
    feats = {}

    def lc_fe(df: pd.DataFrame):
        return __LC_FE(df.index.to_numpy(dtype=np.float64), df["Flux"].to_numpy(), df["Flux_err"].to_numpy()) # pyright: ignore[reportCallIssue]

    # TODO Try optimizing
    flc_fin_df = flc_df[(np.isfinite(flc_df.index) & np.isfinite(flc_df["Flux"]) & np.isfinite(flc_df["Flux_err"]))]
    
    if len(flc_fin_df) >= __LC_FE_MIN_NROWS:
        lc_feats = lc_fe(flc_fin_df)

        for feat_name, feat in zip(__LC_FE.names, lc_feats): # pyright: ignore[reportAttributeAccessIssue]
            feats[feat_name] = feat

    for filter in __FILTERS:
        flc_fin_sub_df = flc_fin_df.loc[flc_df["Filter"] == filter]

        if len(flc_fin_sub_df) >= __LC_FE_MIN_NROWS:
            lc_feats = lc_fe(flc_fin_sub_df)

            for feat_name, feat in zip(__LC_FE.names, lc_feats): # pyright: ignore[reportAttributeAccessIssue]
                feats[f"{feat_name}_{filter}"] = feat

    return feats


def __build_domain_feats_for_obj(flc_df: pd.DataFrame) -> dict:
    return {}


__LC_FE = lc.Extractor(
    lc.LinearFit(), # pyright: ignore[reportArgumentType]
    lc.StetsonK(), # pyright: ignore[reportArgumentType]
    lc.Amplitude(), # pyright: ignore[reportArgumentType]
    lc.BeyondNStd(), # pyright: ignore[reportArgumentType]
    lc.Skew(), # pyright: ignore[reportArgumentType]
    lc.Kurtosis(), # pyright: ignore[reportArgumentType]
)
__LC_FE_MIN_NROWS = 4


# TODO Consider extinction.fitzpatrick99
def __de_extinct(log_row: pd.Series, flc_sub_df: pd.DataFrame):
    r_λ = flc_sub_df["Filter"].map({
        "u": 4.81,
        "g": 3.64,
        "r": 2.70,
        "i": 2.06,
        "z": 1.58,
        "y": 1.31
    })

    c_λ = np.pow(10, 0.4 * r_λ * log_row["EBV"])

    flc_sub_df["Flux"] *= c_λ
    flc_sub_df["Flux_err"] *= c_λ


__build_and_ingest_feats(type="train")
__build_and_ingest_feats(type="test")

Building features for `split_20` (`test`): 100%|██████████| 7135/7135 [02:50<00:00, 41.90obj/s]


### Data Cleaning

In [10]:
train_df = load_all_feats_df(type="train")

X = train_df.drop(columns=["SpecType", "English Translation", "split", "target"])
y = train_df["target"]

X

,Z,Z_err,EBV,Flux_mean,Flux_err_mean,Flux_ratio_mean,Flux_std,Flux_err_std,Flux_ratio_std,Flux_min,Flux_err_min,Flux_ratio_min,Flux_max,Flux_err_max,Flux_ratio_max,Flux_median,Flux_err_median,Flux_ratio_median,Flux_q25,Flux_err_q25,Flux_ratio_q25,Flux_q75,Flux_err_q75,Flux_ratio_q75,linear_fit_slope,linear_fit_slope_sigma,linear_fit_reduced_chi2,stetson_K,amplitude,beyond_1_std,skew,kurtosis,linear_fit_slope_u,linear_fit_slope_sigma_u,linear_fit_reduced_chi2_u,stetson_K_u,amplitude_u,beyond_1_std_u,skew_u,kurtosis_u,linear_fit_slope_g,linear_fit_slope_sigma_g,linear_fit_reduced_chi2_g,stetson_K_g,amplitude_g,beyond_1_std_g,skew_g,kurtosis_g,linear_fit_slope_r,linear_fit_slope_sigma_r,linear_fit_reduced_chi2_r,stetson_K_r,amplitude_r,beyond_1_std_r,skew_r,kurtosis_r,linear_fit_slope_i,linear_fit_slope_sigma_i,linear_fit_reduced_chi2_i,stetson_K_i,amplitude_i,beyond_1_std_i,skew_i,kurtosis_i,linear_fit_slope_z,linear_fit_slope_sigma_z,linear_fit_reduced_chi2_z,stetson_K_z,amplitude_z,beyond_1_std_z,skew_z,kurtosis_z,linear_fit_slope_y,linear_fit_slope_sigma_y,linear_fit_reduced_chi2_y,stetson_K_y,amplitude_y,beyond_1_std_y,skew_y,kurtosis_y
object_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Dornhoth_fervain_onodrim,3.0490,NaN,0.110,1.168616,0.622708,4.149514,5.789107,0.585147,18.428182,-3.147488,0.108253,-11.342471,29.395555,3.305695,106.648350,-0.453209,0.438531,-0.979915,-1.626724,0.237955,-3.941111,1.244752,0.738093,2.469841,0.047048,0.001662,335.340008,0.535145,16.271522,0.061538,3.604880,14.753552,-0.015457,0.017825,7.131294,0.700644,2.137378,0.200000,2.219968,4.941176,-0.017422,0.003350,40.261249,0.713828,1.330206,0.166667,2.234531,5.217528,0.045950,0.002550,287.175000,0.762503,7.773897,0.125000,1.910247,3.894908,-0.034528,0.005263,856.109379,0.673674,15.499414,0.066667,2.860455,9.222710,0.095169,0.009774,368.385780,0.575863,16.206499,0.083333,3.328534,11.318189,0.018636,0.026917,3.581662,0.828991,4.923171,0.090909,1.525677,2.634059
Dornhoth_galadh_ylf,0.4324,NaN,0.058,0.429695,0.587684,1.024884,1.488888,0.561601,3.150463,-1.873722,0.099192,-2.310289,12.200074,3.415717,19.837043,0.102517,0.376465,0.300500,-0.125087,0.228451,-0.399839,0.497571,0.671771,1.272079,-0.007859,0.000434,8.024854,0.537792,7.036898,0.113772,4.293180,26.974214,-0.003246,0.002537,1.837503,0.806376,1.202322,0.400000,0.292289,-0.524390,-0.007588,0.000833,5.259381,0.681003,0.929856,0.125000,1.791914,4.384176,-0.004768,0.000691,8.106536,0.593222,1.728522,0.114286,2.413997,5.661699,-0.010981,0.000924,10.442061,0.528526,3.107300,0.108108,3.836836,17.689467,-0.018654,0.001698,8.386749,0.520283,4.524364,0.114286,3.127142,12.145236,-0.041316,0.005185,5.486354,0.535131,7.036898,0.103448,2.983446,11.688474
Elrim_melethril_thul,0.4673,NaN,0.577,5.218302,1.305302,6.251681,6.401231,0.998163,6.124101,-12.840542,0.485480,-8.692603,15.324461,3.996864,20.016695,5.141539,0.892864,6.227562,2.197429,0.657180,1.746627,9.409224,1.444121,10.377218,0.259221,0.012890,19.580056,0.830733,14.082501,0.285714,-0.701303,0.793689,-0.515701,0.242672,0.166161,0.916153,5.126456,0.400000,-0.479718,-1.765225,0.240841,0.020901,5.219288,0.888096,4.214009,0.444444,0.549476,-1.029628,0.338945,0.025338,4.968273,0.877750,4.748175,0.200000,1.029109,-0.693581,0.261556,0.048382,80.447608,0.726396,11.842347,0.166667,-1.918019,4.062163,0.169156,0.080912,18.318544,0.887082,6.267718,0.375000,0.046864,-1.455541,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ithil_tobas_rodwen,0.6946,NaN,0.012,0.385203,0.428335,1.657185,0.875471,0.472227,2.612474,-7.753266,0.053466,-3.591440,5.431901,3.314667,13.762766,0.339765,0.271678,1.149099,-0.036633,0.169572,-0.131530,0.749992,0.458578,2.699616,-0.000373,0.000027,5.865236,0.725832,6.592584,0.184211,-1.154318,14.607922,-0.000149,0.000107,1.415424,0.814315,1.615229,0.240741,-0.285355,2.166529,-0.000219,0.000038,8.364684,0.747161,0.951293,0.328358,0.541052,0.133894,-0.000776,0.000050,9.619017,0.756481,1.322095,0.302013,0.594775,0.062676,-0.000

### Hyperparameter Tuning with Manual F1 Threshold Optimization

In [11]:
# def __calc_best_f1_score_and_threshold(y: pd.Series, probs):
#     thresholds = np.linspace(0.01, 0.99, 1000)
#     f1_scores = np.array([f1_score(y, (probs > threshold).astype(int)) for threshold in thresholds])

#     idx = f1_scores.argmax()

#     return f1_scores[idx], thresholds[idx]


__SKF = StratifiedKFold(n_splits=10, random_state=42, shuffle=True)
__OPTUNA_SAMPLER = optuna.samplers.TPESampler(seed=42, multivariate=True)

c:\Users\nguye\AppData\Local\pypoetry\Cache\virtualenvs\mallorn-_y-VJO5c-py3.12\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(


In [ ]:
def __cb_objective(trial: optuna.Trial):
    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC", # More smooth convergence than F1
        "verbose": False,
        "allow_writing_files": False,
        "task_type": "CPU", # CPU is faster for 3k rows & required for 'Ordered'
        
        # Crucial for small data (prevents gradient leakage)
        "boosting_type": "Ordered", 
        "bootstrap_type": "Bernoulli", 
        
        # Handle Imbalance
        "auto_class_weights": "Balanced",
        
        # Tuning
        "iterations": trial.suggest_int("cb_iterations", 100, 1000),
        "learning_rate": trial.suggest_float("cb_learning_rate", 1e-3, 0.3, log=True),
        "depth": trial.suggest_int("cb_depth", 3, 8), # Keep shallow for small data
        "l2_leaf_reg": trial.suggest_float("cb_l2_leaf_reg", 1, 10, log=True),
        "subsample": trial.suggest_float("cb_subsample", 0.5, 0.95),
        "random_strength": trial.suggest_float("cb_random_strength", 1e-9, 10, log=True),
    }

    scores = []

    for train_idx, val_idx in __SKF.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        cb_clf = CatBoostClassifier(**params)
        cb_clf.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)
        
        probs = cb_clf.predict_proba(X_val)[:, 1]
        score = average_precision_score(y_val, probs)
        scores.append(score)

    return np.mean(scores)


cb_study = optuna.create_study(direction="maximize", sampler=__OPTUNA_SAMPLER)
cb_study.optimize(__cb_objective, n_trials=50, n_jobs=-1, show_progress_bar=True) # pyright: ignore[reportArgumentType]

[I 2025-12-11 22:04:10,962] A new study created in memory with name: no-name-8b692deb-77ab-42b6-a6e4-21f110b5e455
Best trial: 2. Best value: 0.402482:   2%|▏         | 1/50 [02:20<1:54:36, 140.34s/it]

[I 2025-12-11 22:06:31,295] Trial 2 finished with value: 0.40248168179729477 and parameters: {'cb_iterations': 494, 'cb_learning_rate': 0.19503720826790694, 'cb_depth': 4, 'cb_l2_leaf_reg': 1.3989485973321543, 'cb_subsample': 0.7721160018876858, 'cb_random_strength': 9.44958255085123e-08}. Best is trial 2 with value: 0.40248168179729477.


Best trial: 2. Best value: 0.402482:   4%|▍         | 2/50 [02:58<1:04:20, 80.43s/it] 

[I 2025-12-11 22:07:09,783] Trial 4 finished with value: 0.29655827843114085 and parameters: {'cb_iterations': 942, 'cb_learning_rate': 0.0012143030181593862, 'cb_depth': 3, 'cb_l2_leaf_reg': 2.2470907889862626, 'cb_subsample': 0.6540249053086482, 'cb_random_strength': 3.9940339113984377e-07}. Best is trial 2 with value: 0.40248168179729477.


Best trial: 2. Best value: 0.402482:   6%|▌         | 3/50 [03:17<40:46, 52.05s/it]  

...

[I 2025-12-11 22:09:40,473] Trial 10 finished with value: 0.40209427225806743 and parameters: {'cb_iterations': 810, 'cb_learning_rate': 0.14594120563272334, 'cb_depth': 4, 'cb_l2_leaf_reg': 1.3879034406365425, 'cb_subsample': 0.7145181666760317, 'cb_random_strength': 4.642971181959261e-09}. Best is trial 16 with value: 0.438725614148896.


Best trial: 16. Best value: 0.438726:  16%|█▌        | 8/50 [08:41<58:57, 84.23s/it]

[I 2025-12-11 22:12:52,628] Trial 20 finished with value: 0.3618318256952776 and parameters: {'cb_iterations': 618, 'cb_learning_rate': 0.0045827196929451525, 'cb_depth': 4, 'cb_l2_leaf_reg': 2.400793465967667, 'cb_subsample': 0.8783759795267274, 'cb_random_strength': 4.780226840294942e-06}. Best is trial 16 with value: 0.438725614148896.


Best trial: 16. Best value: 0.438726:  18%|█▊        | 9/50 [10:29<1:02:31, 91.51s/it]

[I 2025-12-11 22:14:40,130] Trial 14 finished with value: 0.35006246811636077 and parameters: {'cb_iterations': 631, 'cb_learning_rate': 0.001103034511595799, 'cb_depth': 6, 'cb_l2_leaf_reg': 5.9320787594493565, 'cb_subsample': 0.9256080786816356, 'cb_random_strength': 0.15832469167215082}. Best is trial 16 with value: 0.438725614148896.


In [ ]:
def __brf_objective(trial: optuna.Trial):
    params = {
        "n_estimators": trial.suggest_int("brf_n_estimators", 100, 800),
        "max_depth": trial.suggest_int("brf_max_depth", 3, 20),
        "min_samples_leaf": trial.suggest_int("brf_min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("brf_max_features", ["sqrt", "log2", None]),
        "sampling_strategy": "all", # Force resampling to achieve balance
        "replacement": True,
        "n_jobs": -1,
        "random_state": 42
    }

    scores = []

    for train_idx, val_idx in __SKF.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        brf_clf = BalancedRandomForestClassifier(**params)
        brf_clf.fit(X_train, y_train)
        
        probs = brf_clf.predict_proba(X_val)[:, 1]
        score = average_precision_score(y_val, probs)
        scores.append(score)

    return np.mean(scores)


brf_study = optuna.create_study(direction="maximize", sampler=__OPTUNA_SAMPLER)
brf_study.optimize(__brf_objective, n_trials=50, n_jobs=-1, show_progress_bar=True) # pyright: ignore[reportArgumentType]

In [ ]:
best_cb_params = cb_study.best_params
best_cb_params.update({
    "loss_function": "Logloss",
    "eval_metric": "F1",
    "verbose": False,
    "allow_writing_files": False,
    "task_type": "CPU",
    "boosting_type": "Ordered", 
    "bootstrap_type": "Bernoulli", 
    "auto_class_weights": "Balanced",
})

best_brf_params = brf_study.best_params
best_brf_params.update({
    "sampling_strategy": "all",
    "replacement": True,
    "n_jobs": -1,
    "random_state": 42
})

cb_oof = np.zeros(len(X))
brf_oof = np.zeros(len(X))

for train_idx, val_idx in __SKF.split(X, y):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    cb_clf = CatBoostClassifier(**best_cb_params)
    cb_clf.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)
    cb_oof[val_idx] = cb_clf.predict_proba(X_val)[:, 1]

    brf_clf = BalancedRandomForestClassifier(**best_brf_params)
    brf_clf.fit(X_train, y_train)
    brf_oof[val_idx] = brf_clf.predict_proba(X_val)[:, 1]


def find_best_ensemble_vectorized(y_true, p_catboost, p_brf, n_weights=101, n_thresholds=1000):
    """
    Finds the optimal weight and threshold simultaneously using 3D broadcasting.
    
    Dimensions Legend:
    - T: Number of Thresholds (axis 0)
    - W: Number of Weights (axis 1)
    - N: Number of Data points (axis 2)
    """
    
    # 1. Define Grids
    # Weights from 0.0 to 1.0
    weights = np.linspace(0, 1, n_weights) # Shape: (W,)
    # Thresholds from 0.01 to 0.99
    thresholds = np.linspace(0.01, 0.99, n_thresholds) # Shape: (T,)

    # 2. Create Weighted Probabilities Matrix
    # Broadcasting: (W, 1) * (N,) -> (W, N)
    # Result: A matrix where every row is a different blend of the models
    p_weighted = (weights[:, None] * p_catboost) + ((1 - weights[:, None]) * p_brf)

    # 3. Create 3D Prediction Tensor
    # We compare (1, W, N) against (T, 1, 1) to generate a (T, W, N) boolean tensor
    # This tensor contains the binary prediction for every single point, 
    # for every weight, for every threshold.
    preds_tensor = p_weighted[None, :, :] >= thresholds[:, None, None]

    # 4. Vectorized Confusion Matrix Calculation
    # Align y_true to (1, 1, N) for broadcasting
    y_broad = y_true[None, None, :].astype(bool)
    
    # Sum over axis 2 (the data points) to get counts for each (Threshold, Weight) pair
    # Logical AND is fast on booleans
    tp = (preds_tensor & y_broad).sum(axis=2)
    fp = (preds_tensor & ~y_broad).sum(axis=2)
    fn = (~preds_tensor & y_broad).sum(axis=2)

    # 5. Compute F1 Grid (T, W)
    # F1 = 2TP / (2TP + FP + FN)
    denominator = 2 * tp + fp + fn
    
    # Safe division to handle cases where model predicts all 0s (denom=0)
    f1_grid = np.divide(
        2 * tp, 
        denominator, 
        out=np.zeros_like(denominator, dtype=float), 
        where=denominator != 0
    )

    # 6. Find the Absolute Maximum
    # argmax gives the flattened index; unravel_index converts it back to (T, W)
    best_idx = np.unravel_index(np.argmax(f1_grid), f1_grid.shape)
    best_t_idx, best_w_idx = best_idx

    return {
        "best_f1": f1_grid[best_idx],
        "best_weight_catboost": weights[best_w_idx], # Weight for p_catboost
        "best_weight_brf": 1 - weights[best_w_idx],  # Weight for p_brf
        "best_threshold": thresholds[best_t_idx]
    }

results = find_best_ensemble_vectorized(y, cb_oof, brf_oof)

print(f"Max F1 Score: {results['best_f1']:.5f}")
print(f"Optimal Mix : {results['best_weight_catboost']:.2f} (CatBoost) / {results['best_weight_brf']:.2f} (BRF)")
print(f"Threshold   : {results['best_threshold']:.4f}")

### Feature Importance Analysis

### Inference

In [ ]:
test_df = load_all_feats_df(type="test")

X_test = test_df.drop(columns=["SpecType", "English Translation", "split"])

X_test

In [ ]:
cb_clf = CatBoostClassifier(**best_cb_params)
cb_clf.fit(X, y, verbose=False)

brf_clf = BalancedRandomForestClassifier(**best_brf_params)
brf_clf.fit(X, y)

In [ ]:
probs_cb = cb_clf.predict_proba(X_test)[:, 1]
probs_brf = brf_clf.predict_proba(X_test)[:, 1]

w_cb = results['best_weight_catboost']
w_brf = results['best_weight_brf']

ensemble_probs = (w_cb * probs_cb) + (w_brf * probs_brf)

best_threshold = results['best_threshold']
test_preds = (ensemble_probs >= best_threshold).astype(int)

In [ ]:
Path("../artifacts/preds").mkdir(parents=True, exist_ok=True)

test_preds_df = pd.DataFrame({"object_id": X_test.index, "target": test_preds})
test_preds_df.to_csv(f"../artifacts/preds/submission-{now()}.csv", index=False)

In [ ]:
# TODO
# Assuming GB, no imputation, no scaling, only minimal cleaning.
# Plot f1 score changes over trials & hyperparams
# Plot feature importance
# Set seeds to 67